# Aprendizagem de Máquina para Web Developers

## Deploy de Modelos de ML — do notebook a um microsserviço

**Formato:** aula remota, ambiente **Google Colab**.

Ao final desta aula você terá:

1. Treinado um pipeline de classificação de texto em **português** (dataset B2W-Reviews01).
2. Serializado o **pipeline inteiro** (vetorizador + modelo) em um único artefato.
3. Empacotado esse modelo em uma **API REST com FastAPI**.
4. Testado a API de verdade — **de dentro do Colab**, sem depender de um segundo terminal.
5. Visto como o mesmo código vira um **projeto de deploy real** (arquivos + Docker + nuvem).

> **Por que esta versão é diferente:** o Colab não tem um "segundo terminal" e o `localhost` dele não é acessível pelo seu navegador. Então, em vez de mandar você "rodar `uvicorn` em outra janela", nós testamos a API **em processo**, com uma ferramenta que chama os endpoints diretamente. Isso funciona para todo mundo, ao mesmo tempo, numa aula remota.


---

## Parte 1 — Por que o notebook não é o ambiente final (conceito)

O desenvolvimento de ML tem duas fases com objetivos opostos:

- **Fase experimental:** exploração, gráficos, tentativa e erro. O notebook é imbatível aqui — células isoladas, resultado imediato.
- **Fase de produção:** engenharia. O objetivo é um serviço **confiável, reproduzível e automatizável**.

As qualidades que tornam o notebook ótimo para experimentar são exatamente o que atrapalha em produção:

- **Estado oculto:** rodar células fora de ordem cria um estado que ninguém consegue reproduzir. Produção exige execução de cima a baixo, sempre igual.
- **Versionamento inviável:** `.ipynb` é um JSON com código, saídas e imagens misturados. O `diff` no Git fica ilegível.
- **Sem práticas de engenharia:** testes, CI/CD, gestão de dependências e tratamento de erro não se encaixam naturalmente no notebook.

**A virada de mentalidade:** sair de um _script interativo_ (notebook) para um _pipeline reproduzível_ (arquivos `.py` que rodam sozinhos). É isso que vamos praticar — inclusive **dentro** deste notebook, deixando claro o tempo todo o que seria um arquivo de projeto.


### 1.1 Serialização: salvar o modelo treinado

Depois de treinado, o modelo é um objeto na memória do Python. Para reusá-lo sem retreinar, **persistimos** esse objeto em disco — isso é _serialização_. Usamos `joblib`, que é mais eficiente que o `pickle` padrão para objetos com arrays NumPy (como os do scikit-learn).

### 1.2 A regra de ouro: **serialize o pipeline INTEIRO, não só o modelo**

O erro clássico: treinar o `TfidfVectorizer` e o classificador separados e salvar **só o classificador**. Isso **falha em produção**. O vetorizador aprendeu um vocabulário e pesos IDF a partir dos dados de treino; se em produção você instancia um vetorizador novo, ele tem outro vocabulário e as previsões viram ruído.

A solução é o **`Pipeline` do scikit-learn**: ele encadeia vetorizador + modelo em **um único objeto**. Salvamos esse objeto. Ele contém tudo — vocabulário, pesos IDF e coeficientes — para ir do **texto cru** à previsão.

> Guarde esta frase: **não se salva o modelo, salva-se o pipeline treinado inteiro.**

### 1.3 Expor via API REST (e por que FastAPI)

Empacotamos o modelo como um **microsserviço**: o front-end não precisa saber Python nem as dependências do modelo — faz um `POST /predict` e recebe JSON. Usamos **FastAPI** porque ele valida a entrada automaticamente (Pydantic), gera documentação interativa em `/docs` e tem ótima performance (ASGI).


---

## Parte 2 — Preparando o dataset (fase experimental)

Vamos usar o **B2W-Reviews01**, um corpus aberto com +130 mil avaliações reais de e-commerce em português. Para a aula usamos uma **amostra de 10 mil linhas**, que baixa em segundos direto por URL — sem login no Kaggle.


In [ ]:
# Baixa a amostra de 10k do B2W-Reviews01 (direto, sem autenticação)
import urllib.request
import os

URL = "https://raw.githubusercontent.com/alan-barzilay/NLPortugues/master/Semana%2003/data/b2w-10k.csv"
if not os.path.exists("b2w-10k.csv"):
  urllib.request.urlretrieve(URL, "b2w-10k.csv")
  print("Download concluído.")
else:
  print("Arquivo já existe.")

# (Base completa, ~130k linhas — opcional, mais lenta:)
# https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/B2W-Reviews01.csv  (separador ';')

Arquivo já existe.


In [2]:
import pandas as pd

df = pd.read_csv("b2w-10k.csv")
print("Formato:", df.shape)
df[["review_text", "overall_rating"]].head()

Formato: (9999, 19)


,review_text,overall_rating
0,Estou contente com a compra entrega rápida o ú...,4
1,"Por apenas R$1994.20,eu consegui comprar esse ...",4
2,SUPERA EM AGILIDADE E PRATICIDADE OUTRAS PANEL...,4
3,MEU FILHO AMOU! PARECE DE VERDADE COM TANTOS D...,4
4,"A entrega foi no prazo, as americanas estão de...",5


### 2.1 Definindo o alvo (target)

Transformamos a nota de 1 a 5 em um problema **binário**:

- notas **4 e 5** → `1` (positivo)
- notas **1 e 2** → `0` (negativo)
- nota **3** → removida (neutra/ambígua)


In [3]:
df = df[["review_text", "overall_rating"]].dropna()
df = df[df["overall_rating"] != 3]  # remove neutros
df["label"] = (df["overall_rating"] >= 4).astype(int)  # 1=positivo, 0=negativo

print(df["label"].value_counts())
print("\nTotal de exemplos:", len(df))

label
1    6247
0    2536
Name: count, dtype: int64

Total de exemplos: 8783


---

## Parte 3 — Treinando e salvando o pipeline

Repare: **um** objeto `Pipeline` cuida da vetorização **e** da classificação. É ele que salvamos.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
  df["review_text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

pipeline = Pipeline(
  [
    ("vectorizer", TfidfVectorizer(min_df=2, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000)),
  ]
)

pipeline.fit(X_train, y_train)
print(classification_report(y_test, pipeline.predict(X_test), digits=3))

              precision    recall  f1-score   support

           0      0.911     0.824     0.865       507
           1      0.931     0.967     0.949      1250

    accuracy                          0.926      1757
   macro avg      0.921     0.896     0.907      1757
weighted avg      0.925     0.926     0.925      1757



In [ ]:
# Testando com frases novas — note que passamos TEXTO CRU direto ao pipeline
for frase in [
  "o produto é ótimo, chegou antes do prazo",
  "veio quebrado e o suporte não respondeu",
  "uma porcaria",
  "show de bola",
]:
  pred = pipeline.predict([frase])[0]
  print(f"{'positivo' if pred == 1 else 'negativo'}  <-  {frase!r}")

positivo  <-  'o produto é ótimo, chegou antes do prazo'
negativo  <-  'veio quebrado e o suporte não respondeu'
negativo  <-  'uma porcaria'
positivo  <-  'show de bola'


In [6]:
import joblib
import os

os.makedirs("models", exist_ok=True)
joblib.dump(pipeline, "models/pipeline.joblib")
print("Pipeline salvo em models/pipeline.joblib")

# Prova de que o artefato é auto-suficiente: carregamos e prevemos a partir de texto cru
carregado = joblib.load("models/pipeline.joblib")
print(carregado.predict(["adorei, recomendo demais"]))  # -> [1]

Pipeline salvo em models/pipeline.joblib
[1]


> **Isto seria um arquivo `train.py`.** Todo o código das Partes 2 e 3 — baixar, preparar, treinar, salvar — é o que num projeto real vive num único script executável `python train.py`. No trabalho final, é exatamente isso que você vai entregar.


---

## Parte 4 — Construindo a API com FastAPI

Agora escrevemos o servidor. Usamos `%%writefile` para criar os arquivos do projeto **a partir do notebook** — deixando explícito que, num deploy real, estes são arquivos versionados no Git, não células.

Nossa API terá o **contrato** (que é também o contrato do trabalho final):

| Rota                               | Entrada           | Saída                                                      |
| ---------------------------------- | ----------------- | ---------------------------------------------------------- |
| `GET /health`                      | —                 | `{"status": "ok"}`                                         |
| `POST /predict`                    | `{"text": "..."}` | `{"label": "positivo"\|"negativo", "confidence": 0.0–1.0}` |
| `POST /predict` com corpo inválido | ex.: `{}`         | erro `422` (automático do Pydantic)                        |


In [ ]:
import os

os.makedirs("app", exist_ok=True)
open("app/__init__.py", "w").close()  # torna 'app' um pacote Python
print("pacote app/ criado")

pacote app/ criado


### 4.1 O arquivo `app/main.py`

Dois pontos de engenharia moderna aqui:

- **`lifespan`** em vez de `@app.on_event("startup")` (que está **deprecado**). O `lifespan` carrega o modelo **uma única vez** quando a API sobe — não a cada requisição.
- **Pydantic** define entrada (`ReviewRequest`) e saída (`PredictionResponse`); a validação `422` sai de graça.


In [ ]:
%%writefile app/main.py
import os, joblib
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

MODEL_PATH = os.getenv("MODEL_PATH", "models/pipeline.joblib")
ml = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
  ml["pipeline"] = joblib.load(MODEL_PATH)  # carrega UMA vez, na inicialização
  yield
  ml.clear()


app = FastAPI(title="Sentiment API", version="1.0.0", lifespan=lifespan)


class ReviewRequest(BaseModel):
  text: str


class PredictionResponse(BaseModel):
  label: str
  confidence: float


@app.get("/health")
def health():
  return {"status": "ok"}


@app.post("/predict", response_model=PredictionResponse)
def predict(req: ReviewRequest):
  pipeline = ml.get("pipeline")
  if pipeline is None:
    raise HTTPException(status_code=503, detail="Modelo indisponível")
  pred = int(pipeline.predict([req.text])[0])
  proba = pipeline.predict_proba([req.text])[0]
  label = "positivo" if pred == 1 else "negativo"
  return PredictionResponse(label=label, confidence=float(max(proba)))


Overwriting app/main.py


---

## Parte 5 — Testando a API **de dentro do Colab**

Aqui está a diferença que faz esta aula funcionar remotamente. Em vez de subir um servidor de rede e abrir `localhost` (que **não funciona** no Colab), usamos o **`TestClient`** do FastAPI: ele chama os endpoints **em processo**, como se fizesse requisições HTTP, mas sem porta nenhuma. É o mesmo mecanismo usado para testar APIs de verdade.


In [ ]:
from app.main import app
from fastapi.testclient import TestClient

with TestClient(app) as client:  # o 'with' dispara o lifespan (carrega o modelo)
  print("GET /health ->", client.get("/health").json())

  r = client.post("/predict", json={"text": "produto excelente, entrega rápida"})
  print("POST /predict (+) ->", r.json())

  r = client.post("/predict", json={"text": "horrível, não recomendo a ninguém"})
  print("POST /predict (-) ->", r.json())

  r = client.post("/predict", json={})  # corpo inválido de propósito
  print("POST /predict inválido -> status", r.status_code)  # esperado: 422

GET /health -> {'status': 'ok'}
POST /predict (+) -> {'label': 'positivo', 'confidence': 0.9907485389344157}
POST /predict (-) -> {'label': 'negativo', 'confidence': 0.8584569656631581}
POST /predict inválido -> status 422


/home/aldemir/Workspace/repositories/web-academy-ufam/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


> **E o Swagger UI (`/docs`)?** Ele existe e é ótimo, mas só aparece no navegador quando a API roda num servidor acessível. No Colab isso exige um **túnel público** — veja o bônus opcional no fim. Para a aula, o `TestClient` acima já prova que a API funciona.


### 5.1 (Bônus opcional) Expor a API com uma URL pública via ngrok

Só se você quiser ver o `/docs` no navegador. Requer criar uma conta grátis no ngrok e colar seu token. **Não é necessário** para o trabalho.


In [10]:
# BÔNUS — opcional. Descomente para rodar.
# !pip install -q pyngrok nest_asyncio uvicorn
# from pyngrok import ngrok, conf
# import nest_asyncio, uvicorn, threading
#
# conf.get_default().auth_token = "COLE_SEU_TOKEN_NGROK_AQUI"
# nest_asyncio.apply()
# url = ngrok.connect(8000)
# print("Swagger UI em:", f"{url}/docs")
# threading.Thread(target=lambda: uvicorn.run(app, port=8000), daemon=True).start()

### 5.2 Cliente `requests` — como outro serviço consumiria a API

Num deploy real (com URL pública), outra aplicação chamaria assim. Trocando `BASE_URL` pela URL do ngrok (ou do seu deploy), este código roda de verdade.


In [11]:
import requests

# BASE_URL = "https://SEU-SUBDOMINIO.ngrok-free.app"   # do bônus, ou do seu deploy
# for texto in ["amei o produto", "veio com defeito"]:
#     r = requests.post(f"{BASE_URL}/predict", json={"text": texto})
#     print(texto, "->", r.json())
print("Descomente e ajuste BASE_URL quando tiver uma URL pública (bônus ou deploy).")

Descomente e ajuste BASE_URL quando tiver uma URL pública (bônus ou deploy).


---

## Parte 6 — Levando para o mundo real: arquivos, Docker e nuvem

O que fizemos no notebook, num projeto de deploy vira **esta estrutura**:

```
meu-projeto/
├── train.py              # Partes 2–3: baixa dados, treina, salva models/pipeline.joblib
├── app/
│   ├── __init__.py
│   └── main.py           # Parte 4: a API FastAPI
├── models/
│   └── pipeline.joblib   # o artefato serializado
├── requirements.txt
├── Dockerfile
└── README.md
```

### 6.1 `requirements.txt`


In [ ]:
%%writefile requirements.txt
fastapi
uvicorn[standard]
scikit - learn
joblib
pandas
requests


Overwriting requirements.txt


### 6.2 `Dockerfile` (leitura)

O Docker empacota API + modelo + dependências numa imagem portátil que roda igual em qualquer lugar. Você **não precisa** rodar isto no Colab; entenda o que cada linha faz — é o que um serviço de nuvem executa por você.


In [13]:
%%writefile Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
# 0.0.0.0 para aceitar conexões de fora do container
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]


Overwriting Dockerfile


### 6.3 Deploy de verdade (fora da aula)

Para colocar no ar com uma URL pública, sem gerenciar servidor, opções com camada gratuita que aceitam este projeto direto de um repositório Git:

- **Hugging Face Spaces** (com SDK Docker) — o mais simples para esta turma: sobe o repositório e ele constrói a imagem sozinho.
- **Render** / **Railway** / **Fly.io** — conectam ao seu GitHub e fazem build a partir do `Dockerfile`.

O fluxo é sempre o mesmo: `git push` → o serviço lê o `Dockerfile` → constrói a imagem → publica a URL. É a continuação natural do que você já tem aqui.


---

## Parte 7 — Trabalho final

O enunciado completo, os critérios de avaliação e o formato de entrega estão no documento **`Trabalho_Deploy.md`** disponibilizado junto com esta aula.

Em resumo, você vai entregar um **repositório** com a estrutura da Seção 6, servindo um classificador treinado no B2W-Reviews01, respeitando **exatamente** o contrato de API da Parte 4 (`/health` e `/predict`). A entrega é corrigida de forma **automatizada**: sua API é executada e testada. Acertar o contrato é a maior parte da nota — leia o enunciado com atenção.
